In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.datasets import imdb
from tensorflow.keras.layers import Input, Dense, SimpleRNN, Embedding
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import load_model

## 1. Loading the dataset

In [2]:
## Vocabulary Size
max_features = 10000

In [3]:
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_features)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [4]:
print(f'Training Data Shape: {X_train.shape}, Training Labels Shape: {y_train.shape}')
print(f'Test Data Shape: {X_train.shape}, Test Labels Shape: {y_test.shape}')

Training Data Shape: (25000,), Training Labels Shape: (25000,)
Test Data Shape: (25000,), Test Labels Shape: (25000,)


In [5]:
# The training data contains the vector one_hot representation for each review
sample_dataset_sample = X_train[0]

In [6]:
sample_label = y_train[0]

In [7]:
print(sample_dataset_sample)

[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]


In [8]:
print(sample_label)

1


In [9]:
## Mapping words index back to words for understanding
word_index=imdb.get_word_index()

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [10]:
reverse_word_index={value: key for key, value in word_index.items()}

In [11]:
decoded_word=' '.join(reverse_word_index.get(i-3,'?') for i in sample_dataset_sample)
decoded_word

"? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what they have done don't you th

## 2. Preprocessing the dataset

In [20]:
max_len = 200
X_train = sequence.pad_sequences(X_train, maxlen=max_len)
X_test = sequence.pad_sequences(X_test,maxlen=max_len)

In [13]:
X_train[0]

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,   

In [14]:
X_test[0]

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,   

## 3. Training RNN Model

In [21]:
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [22]:
model = Sequential([
    Input(shape=(max_len,)),
    Embedding(input_dim=max_features, output_dim=128),
    SimpleRNN(128),
    Dense(1, activation='sigmoid')
])

In [23]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,025 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=1e-3,
    clipnorm=1.0
)

In [26]:
model.compile(optimizer=optimizer,loss='binary_crossentropy',metrics=['accuracy'])

In [27]:
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, callbacks=[early_stopping])

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.5380 - loss: 0.6863 - val_accuracy: 0.5986 - val_loss: 0.6525
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.7312 - loss: 0.5419 - val_accuracy: 0.7674 - val_loss: 0.4958
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.7934 - loss: 0.4492 - val_accuracy: 0.6596 - val_loss: 0.6769
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.8096 - loss: 0.4278 - val_accuracy: 0.7638 - val_loss: 0.5302
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.8497 - loss: 0.3586 - val_accuracy: 0.7334 - val_loss: 0.5678
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.8636 - loss: 0.3277 - val_accuracy: 0.7196 - val_loss: 0.6003
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.8885 - loss: 0.2825 - val_accuracy: 0.7544 - val_loss: 0.6263
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.9013 - loss: 0.2500 - 

## 4. Save the model

In [28]:
model.save("rnn_model.h5")

## 5. Loading model and prediction

In [36]:
rnn_model = load_model("/kaggle/working/rnn_model.h5")

In [37]:
rnn_model.get_weights()

[array([[ 0.04741125,  0.02256932, -0.01317063, ...,  0.01858296,
         -0.01525313,  0.04395642],
        [ 0.00950178,  0.04948395,  0.01493026, ...,  0.04115482,
         -0.01159233, -0.05077843],
        [-0.04414031, -0.03222881,  0.0068404 , ...,  0.02017107,
         -0.01152884, -0.02048364],
        ...,
        [-0.05687201, -0.01876817,  0.00846004, ..., -0.00638146,
          0.06443643,  0.02596198],
        [ 0.03529317, -0.01047275, -0.03184099, ..., -0.07986858,
          0.03707616, -0.0446312 ],
        [ 0.05190392, -0.01802864, -0.05518844, ..., -0.05137736,
          0.05070659,  0.01373606]], dtype=float32),
 array([[-0.19659837, -0.01944265, -0.04381987, ...,  0.09344564,
         -0.10907482, -0.21905951],
        [-0.15811773, -0.08693052, -0.09143486, ...,  0.00835917,
         -0.0849815 , -0.11773345],
        [ 0.25693637, -0.05642273, -0.0512841 , ..., -0.06619113,
          0.13246277,  0.02044564],
        ...,
        [ 0.20025973,  0.11812452, -0.0

In [38]:
# Function to decode reviews
def decode_review(encoded_review):
    return ''.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

# Function to preprocess user input
def preprocess_text(text):
    words = text. lower(). split()
    encoded_review = [word_index.get(word, 2) + 3 for word in words]
    padded_review = sequence.pad_sequences([encoded_review], maxlen=200)
    return padded_review

In [39]:
def predict_sentiment(review):
    preprocessed_input=preprocess_text(review)
    
    prediction=model.predict(preprocessed_input)
    
    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    
    return sentiment, prediction[0][0]

In [40]:
# Example review for prediction
example_review = "This movie was fantastic! The acting was great and the plot was thrilling."

In [41]:
sentiment, score = predict_sentiment(example_review)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 373ms/step


In [42]:
print(f'Review: {example_review}')
print(f'Sentiment: {sentiment}')
print(f'Prediction Score: {score}')

Review: This movie was fantastic! The acting was great and the plot was thrilling.
Sentiment: Positive
Prediction Score: 0.8226459622383118
